In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy import stats
from scipy.stats import (
    norm,
    bernoulli,
    uniform,
    chisquare,
    kstest,
    entropy
)

# ============================================================
# Funciones auxiliares
# ============================================================

def kl_discrete(emp_probs, model_probs, eps=1e-12):
    emp_probs = np.asarray(emp_probs) + eps
    model_probs = np.asarray(model_probs) + eps

    emp_probs /= emp_probs.sum()
    model_probs /= model_probs.sum()

    return entropy(emp_probs, model_probs)

def analyze_continuous(
        data,
        distribution_name,
        distribution,
        params,
        bins=20
):

    data = np.asarray(data)
    data = data[~np.isnan(data)]

    print("\n" + "=" * 70)
    print(distribution_name)
    print("=" * 70)

    # --------------------------------------------------
    # Histograma empírico
    # --------------------------------------------------

    hist_obs, edges = np.histogram(
        data,
        bins=bins
    )

    N = hist_obs.sum()

    # --------------------------------------------------
    # Probabilidades empíricas
    # --------------------------------------------------

    p_emp = hist_obs / N

    # --------------------------------------------------
    # Probabilidades teóricas
    # --------------------------------------------------

    cdf_hi = distribution.cdf(edges[1:], *params)
    cdf_lo = distribution.cdf(edges[:-1], *params)

    p_model = cdf_hi - cdf_lo

    p_model = p_model / p_model.sum()

    # --------------------------------------------------
    # KL
    # --------------------------------------------------

    kl = kl_discrete(p_emp, p_model)

    print(f"KL = {kl:.6f}")

    # --------------------------------------------------
    # Chi-cuadrado
    # --------------------------------------------------

    expected = p_model * N

    mask = expected > 1

    chi2_stat = np.sum(
        (hist_obs[mask] - expected[mask])**2
        / expected[mask]
    )

    df = mask.sum() - len(params) - 1

    p_value = 1 - stats.chi2.cdf(chi2_stat, df)

    print("\nChi-cuadrado")
    print(f"Estadístico = {chi2_stat:.4f}")
    print(f"gl = {df}")
    print(f"p-value = {p_value:.6f}")

    # --------------------------------------------------
    # KS
    # --------------------------------------------------

    ks_stat, ks_p = kstest(
        data,
        distribution.cdf,
        args=params
    )

    print("\nKolmogorov-Smirnov")
    print(f"D = {ks_stat:.4f}")
    print(f"p-value = {ks_p:.6f}")

    # --------------------------------------------------
    # Gráfico
    # --------------------------------------------------

    x = np.linspace(
        data.min(),
        data.max(),
        500
    )

    plt.figure(figsize=(8,4))

    plt.hist(
        data,
        bins=bins,
        density=True,
        alpha=0.6,
        label="Empírica"
    )

    plt.plot(
        x,
        distribution.pdf(x, *params),
        lw=3,
        label="Modelo"
    )

    plt.title(distribution_name)
    plt.legend()
    plt.show()


def analyze_binomial(data):

    print("\n" + "=" * 70)
    print("BINOMIAL / BERNOULLI")
    print("=" * 70)

    data = np.asarray(data)

    N = len(data)

    p_hat = data.mean()

    print(f"p estimado = {p_hat:.4f}")

    counts = pd.Series(data).value_counts().sort_index()

    obs = counts.values

    emp_probs = obs / obs.sum()

    model_probs = np.array([
        1 - p_hat,
        p_hat
    ])

    kl = kl_discrete(
        emp_probs,
        model_probs
    )

    print(f"KL = {kl:.8f}")

    # --------------------------------------------------
    # Chi-cuadrado
    # --------------------------------------------------

    expected = model_probs * N

    chi2_stat = np.sum(
        (obs - expected)**2 / expected
    )

    df = 1

    p_value = 1 - stats.chi2.cdf(
        chi2_stat,
        df
    )

    print("\nChi-cuadrado")
    print(f"Estadístico = {chi2_stat:.6f}")
    print(f"p-value = {p_value:.6f}")

    # --------------------------------------------------
    # KS
    # --------------------------------------------------

    ks_stat, ks_p = kstest(
        data,
        bernoulli(p_hat).cdf
    )

    print("\nKolmogorov-Smirnov")
    print(f"D = {ks_stat:.6f}")
    print(f"p-value = {ks_p:.6f}")

    # --------------------------------------------------
    # Gráfico
    # --------------------------------------------------

    plt.figure(figsize=(5,4))

    plt.bar(
        [0,1],
        emp_probs,
        width=0.4,
        label="Empírica"
    )

    plt.plot(
        [0,1],
        model_probs,
        'ro',
        markersize=10,
        label="Modelo"
    )

    plt.xticks([0,1])
    plt.legend()
    plt.title("Bernoulli")
    plt.show()




: 

In [ ]:
# ============================================================
# 1. NORMAL
# ============================================================

titanic = sns.load_dataset("titanic")

age = titanic["age"].dropna()

mu_hat = age.mean()
sigma_hat = age.std(ddof=1)

print("\nPARAMETROS NORMAL")
print("mu =", mu_hat)
print("sigma =", sigma_hat)

analyze_continuous(
    age,
    "AJUSTE NORMAL - TITANIC AGE",
    norm,
    (mu_hat, sigma_hat)
)



In [ ]:
# ============================================================
# 2. BINOMIAL / BERNOULLI
# ============================================================

survived = titanic["survived"]

analyze_binomial(survived)

# ============================================================
# 3. UNIFORME
# ============================================================

iris = sns.load_dataset("iris")

x = iris["sepal_length"].values

a_hat = x.min()
b_hat = x.max()

print("\nPARAMETROS UNIFORME")
print("a =", a_hat)
print("b =", b_hat)

analyze_continuous(
    x,
    "AJUSTE UNIFORME - IRIS SEPAL LENGTH",
    uniform,
    (a_hat, b_hat - a_hat)
)